In [2]:
%pip install -q --disable-pip-version-check sentence-transformers transformers accelerate torch beir chromadb pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import random
import warnings
import logging

import torch
import pandas as pd
import chromadb

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

warnings.filterwarnings("ignore")

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)

DATA_DIR = "data"
CHROMA_PATH = f"{DATA_DIR}/chroma"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" # smaller model for faster inference during development; can switch to a larger model for better performance

TOP_K = 5

In [17]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

corpus_df = pd.DataFrame.from_dict(corpus, orient="index").reset_index()
corpus_df = corpus_df.rename(columns={"index": "_id"})

queries_df = pd.DataFrame.from_dict(queries, orient="index", columns=["text"]).reset_index()
queries_df = queries_df.rename(columns={"index": "_id"})

print("Corpus size:", len(corpus))
print("Queries size:", len(queries))
print("Qrels size:", len(qrels))

corpus_df.head()

  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 67638.70it/s]

Corpus size: 5183
Queries size: 300
Qrels size: 300


,_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...


In [18]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(name="scifact")

print("Chroma collection count:", collection.count())

if collection.count() == 0:
    raise ValueError(
        "Chroma collection is empty. Run Part 1 first or copy your data/chroma folder into this project."
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4419.66it/s]


Chroma collection count: 5183


In [19]:
def retrieve_docs(query_text, top_k=5):
    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    retrieved = []

    for rank in range(len(results["ids"][0])):
        doc_id = results["ids"][0][rank]
        doc = corpus[doc_id]

        retrieved.append({
            "rank": rank + 1,
            "id": doc_id,
            "title": doc["title"],
            "text": doc["text"],
            "distance": results["distances"][0][rank] if "distances" in results else None,
        })

    return retrieved

In [20]:
sample_query_id = list(queries.keys())[0]
sample_query = queries[sample_query_id]

retrieved_docs = retrieve_docs(sample_query, top_k=TOP_K)

print("Query ID:", sample_query_id)
print("Query:", sample_query)
print()

for doc in retrieved_docs:
    print("=" * 80)
    print("Rank:", doc["rank"])
    print("Doc ID:", doc["id"])
    print("Title:", doc["title"])
    print(doc["text"][:700])

Query ID: 1
Query: 0-dimensional biomaterials show inductive properties.

Rank: 1
Doc ID: 29638116
Title: Complex Tissue and Disease Modeling using hiPSCs.
Defined genetic models based on human pluripotent stem cells have opened new avenues for understanding disease mechanisms and drug screening. Many of these models assume cell-autonomous mechanisms of disease but it is possible that disease phenotypes or drug responses will only be evident if all cellular and extracellular components of a tissue are present and functionally mature. To derive optimal benefit from such models, complex multicellular structures with vascular components that mimic tissue niches will thus likely be necessary. Here we consider emerging research creating human tissue mimics and provide some recommendations for moving the field forward.
Rank: 2
Doc ID: 4346436
Title: Nonlinear Elasticity in Biological Gels
Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Device: cpu


Loading weights: 100%|██████████| 338/338 [00:11<00:00, 29.90it/s]


In [22]:
test_prompt = """
You are a careful scientific assistant.

Question:
Does aspirin reduce heart attack risk?

Answer in one short paragraph.
"""

output = generator(
    test_prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
)

print(output[0]["generated_text"])

Yes, according to the available evidence, taking low-dose aspirin can help reduce the risk of having a heart attack. This is because aspirin inhibits platelet aggregation, which prevents blood clots from forming and blocking arteries. Studies have shown that people who regularly take small doses of aspirin (typically 75-325 mg per day) may experience a lower incidence of cardiovascular events such as heart attacks compared to those who do not use it. However, it's important to note that while aspirin has been shown to be effective for reducing heart attack risk, its benefits should


In [23]:
def truncate_text(text, max_chars=1200):
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."


def format_docs_for_prompt(retrieved_docs, max_chars_per_doc=1200):
    blocks = []

    for doc in retrieved_docs:
        block = f"""
[Document {doc["rank"]}]
Doc ID: {doc["id"]}
Title: {doc["title"]}
Text:
{truncate_text(doc["text"], max_chars=max_chars_per_doc)}
"""
        blocks.append(block)

    return "\n".join(blocks)


def build_rag_prompt(claim, retrieved_docs):
    evidence = format_docs_for_prompt(retrieved_docs)

    prompt = f"""
You are a scientific claim-checking assistant.

Use ONLY the evidence below. Do not use outside knowledge.

Claim:
{claim}

Evidence:
{evidence}

Instructions:
1. Decide whether the claim is Supported, Refuted, or Unclear based only on the evidence.
2. Explain your reasoning briefly.
3. Cite the document IDs you used.
4. If the evidence is insufficient, say Unclear.

Return your answer in this exact format:

Label: Supported / Refuted / Unclear
Reasoning: ...
Evidence used: ...
"""

    return prompt

In [12]:
def generate_rag_answer(claim, top_k=5, max_new_tokens=220):
    retrieved_docs = retrieve_docs(claim, top_k=top_k)
    prompt = build_rag_prompt(claim, retrieved_docs)

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )

    answer = output[0]["generated_text"].strip()

    return {
        "claim": claim,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": answer,
    }

In [27]:
sample_query_id = list(queries.keys())[0]
sample_claim = queries[sample_query_id]

rag_result = generate_rag_answer(sample_claim, top_k=TOP_K)

print("Query ID:", sample_query_id)
print("Claim:", rag_result["claim"])
print()
print("Generated answer:")
print(rag_result["answer"])
print()
print("=" * 80)
print("Retrieved docs:")
for doc in rag_result["retrieved_docs"]:
    print(f'Rank {doc["rank"]} | Doc ID {doc["id"]} | {doc["title"]}')

Query ID: 1
Claim: 0-dimensional biomaterials show inductive properties.

Generated answer:
Let's analyze the claim "0-dimensional biomaterials show inductive properties" against the provided evidence:

### Claim Analysis:
The claim suggests that 0-dimensional biomaterials exhibit inductive properties. Inductive properties refer to the ability of a substance to initiate or guide other substances towards a particular state or direction.

### Evidence Review:
1. **Document 1**: Discusses the use of human pluripotent stem cells for complex tissue modeling. It mentions the importance of incorporating vascular components to create more realistic tissue mimics. However, it does not mention any specific role of 0-dimensional biomaterials in inducing processes.
   
2. **Document 2**: Focuses on the elasticity of biological materials and the molecular basis of strain stiffening. While it discusses the concept of inductive properties related to elasticity, it does not specifically address 0-dime

In [28]:
def get_gold_doc_ids(query_id):
    return set(qrels.get(query_id, {}).keys())


def retrieval_succeeded(query_id, retrieved_docs):
    retrieved_ids = {doc["id"] for doc in retrieved_docs}
    gold_ids = get_gold_doc_ids(query_id)

    return len(retrieved_ids & gold_ids) > 0


def gold_doc_rank(query_id, retrieved_docs):
    gold_ids = get_gold_doc_ids(query_id)

    for doc in retrieved_docs:
        if doc["id"] in gold_ids:
            return doc["rank"]

    return None

In [29]:
print("Gold doc IDs:", get_gold_doc_ids(sample_query_id))
print("Retrieval succeeded:", retrieval_succeeded(sample_query_id, rag_result["retrieved_docs"]))
print("Gold doc rank:", gold_doc_rank(sample_query_id, rag_result["retrieved_docs"]))

Gold doc IDs: {'31715818'}
Retrieval succeeded: True
Gold doc rank: 5
